In [5]:
import requests
from bs4 import BeautifulSoup
import itertools
from matplotlib.patches import RegularPolygon
from IPython.display import Markdown, display,Image,HTML

import sympy as sp

In [6]:
url='https://www.janestreet.com/puzzles/robot-road-trip-index/'
res = requests.get(url)
soup = BeautifulSoup(res.content, 'html.parser')
x =[text for text in soup.body.stripped_strings]
#display([(i,j) for i,j in enumerate(x)])
display(Markdown(" ".join(x[16:57])))


Robot cars have a top speed (which they prefer to maintain at all times while driving)
that’s a real number randomly drawn uniformly between 1 and 2 miles per minute. A two-lane
highway for robot cars has a fast lane (with minimum speed a ) and a slow lane (with
maximum speed a ). When a faster car overtakes a slower car in the same lane, the slower
car is required to decelerate to either change lanes (if both cars start in the fast lane)
or stop on the shoulder (if both cars start in the slow lane). Robot cars decelerate and
accelerate at a constant rate of 1 mile per minute per minute, timed so the faster,
overtaking car doesn’t have to change speed at all, and passing happens
instantaneously. If cars rarely meet (so you never have to consider a car meeting more
than one other car on its trip, see Mathematical clarification below), and you want to
minimize the miles not driven due to passing, what should a be set to, in miles per
minute? Give your answer to 10 decimal places . Example car interactions: suppose a is set to 1.2 miles per minute. If a car with top
speed 1.8 overtakes a car with top speed 1.1, neither has to slow down because they are in
different lanes. If instead the car with top speed 1.8 overtakes one with top speed 1.7,
the slower car computes the optimal time to start decelerating for 30 seconds (to reach
1.2 miles per minute to switch to the other lane) so the faster car instantly passes and
the slower car can immediately start accelerating for another 30 seconds to return to 1.7
miles per minute. This pass cost 0.25 miles (how far behind where the slower car would be
if it continued at 1.7 miles per minute). If a car with top speed 1.1 overtakes one with top speed 1.0 in the slow lane, the slower
(slowest!) car must decelerate for a full minute all the way to 0 to allow the pass, and
then accelerate for a full minute to reestablish its speed, losing exactly 1 mile of
distance. Assume all car trips are of constant length N , starting at arbitrary points and times
along an infinitely long highway. This is made more mathematically precise below. Mathematical clarification : Say car trips arrive at a rate of z car trip beginnings
per mile per minute, uniformly across the infinite highway (cars enter and exit their
trips at their preferred speed due to on/off ramps), and car trips have a constant length
of N miles. Define f ( z , N ) to be the value of a that minimizes the expected lost
distance per car trip due to passing. Find: the limit of [the limit of f ( z , N ) as z -> 0+] as N -> infinity.

In [36]:
# Set up symbols
s, f, a, N= sp.symbols('s f a N', positive=True, real=True) # speed of slow car,speed of fast car, speed limit


# Meeting rate (hard bit)
# cars spawn at a rate of z per min per mile
# i..e. evenly in space AND time
# if fast car spawns at time 0 at point 0. the relevant time window when both are live is from -N/s to N/f
# and the distance window -N to N
# the area where overtaking happens is a paralleogram

total_area = 2*N*(N/f+N/s)

A = sp.Point(-N, -N/s) 
B = sp.Point(0, 0) 
C = sp.Point(N, N/f)
D = sp.Point(N*(1/f-1/s), 0)
parallelogram = sp.Polygon(A, B, C, D)
overtake_area = sp.simplify(parallelogram.area)

rate = overtake_area/total_area

# Define loss (easy bit)
slow_loss = s**2           # v <  a   (both in slow lane)
fast_loss = (s - a)**2     # u >  a   (both in fast lane)

# Expected loss integrate over fast then slow
slow_total= sp.integrate(sp.integrate(slow_loss * rate, (f, s, a)),(s, 1, a))   # both less than a
fast_total = sp.integrate(sp.integrate(fast_loss * rate, (f, s, 2)),(s, a, 2))  # both greater than a

L = sp.simplify(slow_total + fast_total)
display( L)    

#First derivative to get the optimal speed limit and then solve.

dL = sp.diff(L, a).simplify()
root = sp.nsolve(dL, 1.5)        # initial guess 1.5 

# 6 · Output
print("Optimal a ≈ {:.10f}".format(root))


a**3/4 - a**2/4 - 5*a/8 + log(2**(a**2*(a + 6)/12)*a**(-a**3/12 - a**2/2 + 1/12)) + 2/9

Optimal a ≈ 1.1771414168


In [38]:
url='https://www.janestreet.com/puzzles/robot-road-trip-solution/'
res = requests.get(url)
soup = BeautifulSoup(res.content, 'html.parser')
x =[text for text in soup.body.stripped_strings]
#display([(i,j) for i,j in enumerate(x)])
display(Markdown(" ".join(x[16:54])))

The trickiest part of this probability puzzle is determining the relative base rates
between cars of different speeds overtaking each other. The notes about how cars rarely
meet and all trips are the same long length N (made precise by taking limits) implies
that we can minimize the “first order term” of the cost of exactly two cars meeting and
we can ignore the boundary effects of meeting near the beginning or end of one of the
cars’ journeys. We know each car’s speed is uniformly chosen in [1,2], so to find the
relative rates of overtaking we can integrate over two speeds being drawn and then
consider the set of positions in distance and time (which we are also given as being
uniform) that the slower car would’ve had to start at to be overtaken by the faster car. So say the two speeds we select are u < v , and without loss of generality we fix the
position of the car traveling v to enter the highway at (0, 0) (the first coordinate is
in distance and the second is in time). Then the car will exit the highway at ( N , N/v ), and the starting positions of the u -speed car that need to be overtaken by the v -speed car are those within the pictured parallelogram (again we are ignoring weirdness
along the boundary because it will go to zero in the limit). A few different methods can
compute the area of this parallelogram to be N 2 (1/ u - 1/ v ). There are other ways
to reason out this base rate but I like drawing parallelograms. Now we must compute the cost of each overtake as a function of the speed of the overtaken
car, it turns out to be in proportion to the square of the difference between u and the
speed the car must reduce to. This implies we are trying to choose a to minimize the sum
of the integrals pictured. The first integral is the cost of reducing speed from the slow
lane to zero and the second integral is the cost of reducing speed from the fast lane to
the slow lane. The N 2 factors out and can be ignored, and some fundamental
theorem of calculus and integration computing can get to an expression in a whose
solution is 1.1771414168 … Congrats to all the solvers able to find the optimal speed limit!